# Task 3, Step 4: On-Policy Distillation (Student-Generated Prefixes)

Independent students from the original checkpoint with the same LoRA initialisation rule, update settings, prompt
schedule and 30,000 + 10,000 teacher-scored-token quotas as off-policy. Each presentation samples a fresh response from
the *current* student (temperature 1, top-p 1, caps 192 / 32, no gradient through generation), the teacher scores
those prefixes, and the same forward-KL objective is optimised. Student rollout time is recorded separately; the
number of optimizer updates may differ from off-policy because response lengths differ (reported).

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task3_speculative")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2" or _ver("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5", "peft>=0.13"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task3", need_public=True, need_prefix_banks=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from shared.src.io import read_jsonl, load_json, save_json
from shared.src.plotting import save_figure
from task3_speculative.src.distill import load_models, train

M = load_models(load_draft=False)
tok, target = M["tok"], M["target"]
RUNS = ctx.checkpoints / "distill"
summaries = [train("on", seed, target, tok, RUNS) for seed in (6304, 6305)]
pd.DataFrame([{k: v for k, v in s.items() if k not in ("lora", "opt", "quota", "caps")} for s in summaries])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for method in ("off", "on"):
    for s in (6304, 6305):
        f = RUNS / f"{method}_policy_s{s}" / "updates.jsonl"
        if f.exists():
            u = pd.DataFrame(read_jsonl(f))
            ax[0].plot(u.scored_math + u.scored_short, u.kl, label=f"{method}-policy s{s}")
            p = pd.DataFrame(read_jsonl(RUNS / f"{method}_policy_s{s}" / "presentations.jsonl"))
            ax[1].hist(p[p.domain == "math"].n_scored, bins=30, alpha=0.4, label=f"{method} s{s}")
ax[0].set_xlabel("teacher-scored response tokens"); ax[0].set_ylabel("per-token KL (update mean)"); ax[0].legend()
ax[1].set_xlabel("math response length (scored tokens)"); ax[1].legend()
save_figure(ctx.figures, "distillation_curves_off_vs_on")

In [ ]:
cost = []
for method in ("off", "on"):
    for s in (6304, 6305):
        sm = load_json(RUNS / f"{method}_policy_s{s}" / "summary.json")
        save_json(ctx.results / "distill" / f"{method}_policy_s{s}_summary.json", sm)
        cost.append(dict(run=sm["run"], updates=sm["n_updates"], skipped=sm["n_skipped_updates"], presentations=sm["n_presentations"],
                         scored_math=sm["scored"]["math"], scored_short=sm["scored"]["short"], wall_s=sm["wall_time_s"],
                         **{k: v / 1e3 for k, v in sm["cost_ms"].items()}, peak_mem_mb=sm["peak_mem_mb"],
                         trainable=sm["trainable_params"], total=sm["total_params"]))
pool = load_json(ctx.results / "distill" / "teacher_pool_stats.json")
cdf = pd.DataFrame(cost)
cdf.loc[cdf.run.str.startswith("off"), "teacher_pool_generation_s"] = pool["teacher_generation_s"]
cdf.to_csv(ctx.results / "distill" / "training_costs.csv", index=False)
cdf